# Experiment 03 — Physics-Informed Tucker FrNO (Rank-1 Push)

This notebook now runs the **physics-informed rank-push pipeline** built around a Tucker-factorized fractional residual neural operator.
All reusable code lives in `src/`.

**What changed:**
- Replaced global min-max training with **grid-wise log-standardization** over 2016 data
- Reduced the feature set to the strongest physical drivers: `cpm25`, `pblh`, `t2`, `u10`, `v10`, `q2`, `rain`, `PM25`, `NOx`
- Added diurnal auxiliaries: `hour_sin`, `hour_cos`
- Switched the model to **PhysicsFrNO** with Tucker-factorized spectral weights, learnable fractional phase rotation, and a residual persistence gate
- Switched training to the 3-phase protocol: global stability → tail sharpening → autoregressive RNO fine-tuning
- Kept TTA + checkpoint ensembling in inference, with explicit non-negativity clipping before saving

**Pipeline:**  
1. Load config  
2. Build / load the 2016 grid scaler  
3. Build memory-mapped train/validation loaders  
4. Train PhysicsFrNO with the 3-phase schedule  
5. Run autoregressive inference with TTA / optional checkpoint ensemble and save `preds.npy`

In [ ]:
import sys, os

# ── Path resolution: works both locally and on Kaggle ──
KAGGLE_SRC_DATASET = "ronit-pm25-src"
KAGGLE_DATA_ROOT = "/kaggle/input/datasets/khushisingh942004/aisehack"

if os.path.exists('/kaggle'):
    os.environ['AISEHACK_DATA'] = KAGGLE_DATA_ROOT

LOCAL_SRC = os.path.abspath('../')

_kaggle_candidates = [
    f'/kaggle/input/{KAGGLE_SRC_DATASET}',
    f'/kaggle/input/datasets/ronitraj1/{KAGGLE_SRC_DATASET}',
]
KAGGLE_SRC = next(
    (p for p in _kaggle_candidates if os.path.exists(os.path.join(p, 'src'))),
    _kaggle_candidates[0],
)

SRC_ROOT = KAGGLE_SRC if os.path.exists('/kaggle') else LOCAL_SRC
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f"SRC_ROOT: {SRC_ROOT}")
print(f"AISEHACK_DATA: {os.environ.get('AISEHACK_DATA', 'not set')}")

from src.config import load_config
from src.utils import seed_everything, print_device_info, count_parameters, sanity_check_bounds
from src.data import build_grid_stats, describe_grid_stats, load_grid_stats, load_minmax_bounds, load_all_months, get_dataloaders
from src.model import build_model
from src.train import train, init_wandb, finish_wandb, PERSISTENCE_RMSE_PHYS
from src.inference import run_inference


In [ ]:
import wandb

# ── W&B login ──────────────────────────────────────────────────────────────────
# Pass your API key directly, via WANDB_API_KEY env var, or leave blank to use
# the key already stored by `wandb login` on the CLI.
# On Kaggle: add your key as a Secret named WANDB_API_KEY and it will be picked
# up automatically.
wandb.login(key=os.environ.get("WANDB_API_KEY", None), relogin=False)
print("W&B login complete.")


## 1. Configuration

In [ ]:
cfg = load_config()

seed_everything(cfg['training']['seed'])
print_device_info(cfg['device'])

print(f"Base features ({cfg['features']['n_features']}): {cfg['features']['base']}")
print(f"Aux features  ({len(cfg['features']['aux'])}): {cfg['features']['aux']}")
print(f"Input channels: {cfg['features']['input_channels']}")
print(f"Train months: {cfg['data']['train_months']}")
print(f"Val month:    {cfg['data']['val_month']}")
print(f"Model type:   {cfg['model']['type']}")
print(f"Grid scaler:  {cfg['paths']['grid_stats']}")
print(f"Phase epochs: {cfg['training']['phase1_epochs']} + {cfg['training']['phase2_epochs']} + {cfg['training']['phase3_epochs']}")
print(f"Data root:    {cfg['paths']['data']}")

## 2. Build / Load Grid-Wise Log Standardization Maps

This replaces the old global min-max training normalization.
For each feature and each grid cell, the pipeline computes 2016 mean / std on the transformed domain:
- non-negative features: `log(1 + x)`
- signed features such as `u10`, `v10`: sign-preserving `log(1 + |x|)`

The saved scaler file is reused for train, validation, and test inference.

In [ ]:
# Load official bounds for inverse transforms / diagnostics
bounds = load_minmax_bounds(cfg)
sanity_check_bounds(bounds, cfg['features']['base'])

# Build the per-grid log scaler once, then reuse it
if not os.path.exists(cfg['paths']['grid_stats']):
    grid_stats = build_grid_stats(cfg, bounds=bounds)
else:
    grid_stats = load_grid_stats(cfg)

cfg.setdefault('_runtime', {})['grid_stats'] = grid_stats
print(f"Loaded grid scaler for {len(grid_stats)} features")
print(f"Source: {cfg['paths']['grid_stats']}")
describe_grid_stats(grid_stats, cfg['features']['base'])

## 3. Load & Preprocess Training / Validation Data

Preprocessing used here:
- 2016 grid-wise log-standardization
- memory-mapped month arrays (`mmap_mode='r'`) for low-RAM batching
- strict temporal blocking with Protocol P1 (`OCT_16` validation)
- DEC oversampling via the weighted sampler
- two explicit diurnal channels: `hour_sin`, `hour_cos`

In [ ]:
print("Loading + normalizing training months ...")
train_data = load_all_months(cfg, cfg['data']['train_months'], bounds)

print("\nLoading + normalizing validation month ...")
val_data = load_all_months(cfg, [cfg['data']['val_month']], bounds)

In [ ]:
train_dl, val_dl = get_dataloaders(cfg, train_data, val_data, bounds)
print("Batch shape check ...")
xb, yb = next(iter(train_dl))
print(f"  x: {tuple(xb.shape)}  (B, C={xb.shape[1]}, T={xb.shape[2]}, H={xb.shape[3]}, W={xb.shape[4]})")
print(f"  y: {tuple(yb.shape)}  (B, H, W, T_out={yb.shape[3]})")
print(f"  x range: [{xb.min():.3f}, {xb.max():.3f}]")
print(f"  y range: [{yb.min():.3f}, {yb.max():.3f}]")
print("  PhysicsFrNO mode: grid-standardized channels + autoregressive-ready cpm25 history masking")

## 4. Build & Train PhysicsFrNO

In [ ]:
model = build_model(cfg)
print(f"Parameters: {count_parameters(model):,}")
print(f"Using model: {cfg['model']['type']}")
print(f"Width / modes / depth: {cfg['model'].get('width')} / {cfg['model'].get('modes')} / {cfg['model'].get('depth')}")
print(f"Tucker rank ratio: {cfg['model'].get('rank_ratio', 0.4)}")
print(f"FrFT alpha init:   {cfg['model'].get('alpha_init', 0.35)}")
print(f"AMP enabled:       {cfg['training'].get('use_amp', False)}")
print(f"Checkpoint metric: {cfg['training'].get('checkpoint_metric', 'val_rmse_phys')}")
print(f"Physics enabled:   {cfg.get('physics', {}).get('enabled', False)}")
print(f"Lambda_d:          {cfg['training'].get('lambda_d', 1.0)}")
print(f"Lambda_p:          {cfg['training'].get('lambda_p', 0.0)}")
print(f"Autoreg inference: {cfg.get('inference', {}).get('use_autoregressive', True)}")

In [ ]:
# ── Initialize W&B run (respects cfg['wandb']['enabled']) ─────────────────────
wandb_run = init_wandb(cfg)

history = train(cfg, model, train_dl, val_dl, bounds=bounds, wandb_run=wandb_run)

# ── Finalize W&B: log summary table + best-epoch stats ────────────────────────
finish_wandb(wandb_run, history, bounds, cfg)


### Training Curves & Physics Diagnostics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

ax = axes[0]
ax.plot(history['train_loss'], label='Train RMSE (std-space)', alpha=0.8)
ax.plot(history['val_loss'], label='Val RMSE (std-space)', alpha=0.9)
ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
ax.set_title('Standardized-Space RMSE')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(history['val_phys_rmse'], label='Val RMSE (µg/m³)', alpha=0.9)
ax.plot(history['val_persistence_phys'], label='Val persistence (µg/m³)', alpha=0.8)
ax.axhline(PERSISTENCE_RMSE_PHYS, color='red', linestyle='--', label=f'Global persistence ({PERSISTENCE_RMSE_PHYS:.2f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE (µg/m³)')
ax.set_title('Physical-Space Validation RMSE')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(history['train_objective'], label='Train objective', alpha=0.8)
ax.plot(history['val_objective'], label='Val objective', alpha=0.9)
ax.plot(history['selection_metric'], label='Selection metric', alpha=0.9)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('3-Phase Objective / Selection')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4.1 Schedule Recommendation

This inspects the completed 3-phase run and answers:
- keep the default `20 + 20 + 10` schedule
- extend **phase 2** if the tail-sensitive objective is still improving
- extend **phase 3** if the autoregressive physical RMSE is still dropping near the end

In [ ]:
import numpy as np

val_phys = np.asarray(history['val_phys_rmse'], dtype=float)
val_obj = np.asarray(history['val_objective'], dtype=float)
epochs_ran = len(val_phys)
current_epochs = int(cfg['training']['epochs'])
phase1 = int(cfg['training']['phase1_epochs'])
phase2 = int(cfg['training']['phase2_epochs'])
phase3 = int(cfg['training']['phase3_epochs'])
best_epoch = int(np.argmin(val_phys)) + 1
best_val = float(val_phys[best_epoch - 1])

lookback = min(5, epochs_ran)
tail = val_phys[-lookback:]
trend = float(np.polyfit(np.arange(lookback), tail, 1)[0]) if lookback >= 2 else 0.0
tail_std = float(np.std(val_phys[-min(8, epochs_ran):])) if epochs_ran >= 2 else 0.0

best_near_end = best_epoch >= max(1, epochs_ran - 2)
still_improving = trend < 0

suggest_phase2 = phase2
suggest_phase3 = phase3
reason = 'Keep the default 3-phase schedule.'

if best_near_end and still_improving:
    if epochs_ran <= phase1 + phase2:
        suggest_phase2 = max(phase2, 25)
        reason = 'Validation RMSE is still falling during the tail-sharpening window; extend phase 2 first.'
    else:
        suggest_phase3 = max(phase3, 15)
        reason = 'Autoregressive fine-tuning is still helping near the end; extend phase 3.'
elif best_epoch <= phase1:
    reason = 'The model peaked early; reduce capacity or regularize more instead of training longer.'

print('Schedule recommendation')
print('-' * 32)
print(f'Current schedule: phase1={phase1}, phase2={phase2}, phase3={phase3} (total={current_epochs})')
print(f'Epochs actually ran: {epochs_ran}')
print(f'Best epoch:         {best_epoch}')
print(f'Best val RMSE:      {best_val:.4f} µg/m³')
print(f'Tail trend slope:   {trend:+.6e}')
print(f'Tail std:           {tail_std:.6e}')
print()
print(f'Recommendation: {reason}')
print(f'Suggested phase2 epochs: {suggest_phase2}')
print(f'Suggested phase3 epochs: {suggest_phase3}')

if 'wandb_run' in globals() and wandb_run is not None and wandb.run is not None:
    wandb.run.summary.update({
        'recommendation/current_epochs': current_epochs,
        'recommendation/epochs_ran': epochs_ran,
        'recommendation/best_epoch': best_epoch,
        'recommendation/best_val_rmse_phys': best_val,
        'recommendation/tail_trend': trend,
        'recommendation/tail_std': tail_std,
        'recommendation/suggested_phase2_epochs': suggest_phase2,
        'recommendation/suggested_phase3_epochs': suggest_phase3,
        'recommendation/reason': reason,
    })
    print('Saved recommendation summary to W&B.')


## 5. Autoregressive Inference, TTA & Submit

In [ ]:
import torch

model.load_state_dict(torch.load(cfg['paths']['model_save'], map_location=cfg['device']))
print(f"Loaded checkpoint: {cfg['paths']['model_save']}")
print(f"Autoregressive rollout: {cfg.get('inference', {}).get('use_autoregressive', True)}")
print(f"Inference TTA modes: {cfg.get('inference', {}).get('tta_modes', ['identity'])}")
if cfg.get('inference', {}).get('model_paths'):
    print(f"Ensemble checkpoints: {len(cfg['inference']['model_paths'])}")
preds = run_inference(cfg, model, bounds)
print('Done!')

In [ ]:
import os
import numpy as np

# ------------------------------
# Environment-aware path helpers
# ------------------------------
def first_existing(paths):
    return next((p for p in paths if p and os.path.exists(p)), None)

on_kaggle = os.path.exists('/kaggle')

# Resolve source/project root
if 'SRC_ROOT' in globals() and isinstance(SRC_ROOT, str):
    src_root = SRC_ROOT
else:
    cwd = os.getcwd()
    candidate_roots = [
        os.path.abspath(os.path.join(cwd, '..')),
        os.path.abspath(os.path.join(cwd, '../..')),
        os.path.abspath(cwd),
        os.path.abspath('/home/raj/Documents/CODING/Hackathon/ANRF_AISEHack_Code/Ronit'),
        '/kaggle/input/ronit-pm25-src',
    ]
    src_root = first_existing([r for r in candidate_roots if os.path.exists(os.path.join(r, 'outputs'))]) or candidate_roots[0]

# Resolve data root (works for local + Kaggle competition/input variants)
if 'cfg' in globals() and isinstance(cfg, dict) and 'paths' in cfg:
    cfg_output = cfg['paths'].get('output')
    cfg_model_save = cfg['paths'].get('model_save', '')
    cfg_temp = cfg['paths'].get('temp', '')
    data_root = cfg['paths'].get('data')
else:
    cfg_output = None
    cfg_model_save = ''
    cfg_temp = ''
    data_root = None

data_candidates = [
    data_root,
    os.environ.get('AISEHACK_DATA'),
    '/kaggle/input/aisehack-theme-2',
    '/kaggle/input/competitions/aisehack-theme-2',
    '/kaggle/input/datasets/khushisingh942004/aisehack',
    os.path.abspath(os.path.join(src_root, '..', 'aisehack-theme-2')),
    os.path.abspath(os.path.join(src_root, 'aisehack-theme-2')),
]
data_root = first_existing([p for p in data_candidates if p and os.path.exists(os.path.join(p, 'test_in'))])
if data_root is None:
    raise FileNotFoundError('Could not locate data root containing test_in/. Set AISEHACK_DATA or attach competition data.')

# ------------------------------
# Locate preds.npy
# ------------------------------
pred_candidates = [
    '/kaggle/working/preds.npy' if on_kaggle else None,
    cfg_output,
    os.path.join(os.path.dirname(cfg_model_save), 'preds.npy') if cfg_model_save else None,
    os.path.join(os.path.dirname(cfg_temp), 'preds.npy') if cfg_temp else None,
    os.path.abspath(os.path.join(src_root, 'outputs', 'models', 'preds.npy')),
    os.path.abspath(os.path.join(src_root, 'outputs', 'submissions', 'preds.npy')),
]
preds_path = first_existing(pred_candidates)
if preds_path is None:
    raise FileNotFoundError('Could not find preds.npy in expected locations (including /kaggle/working).')

preds = np.load(preds_path)
print(f"Using preds: {preds_path}")
print(f"preds shape (raw): {preds.shape}")

# ------------------------------
# Load available test history
# ------------------------------
test_cpm25_path = os.path.join(data_root, 'test_in', 'cpm25.npy')
test_cpm25_hist = np.load(test_cpm25_path)
print(f"Using test history: {test_cpm25_path}")
print(f"test_in/cpm25 shape: {test_cpm25_hist.shape}")

# Expected:
# test_cpm25_hist: (N, 10, H, W)
# preds:           (N, H, W, 16)

# Handle common prediction layout variant: (N, 16, H, W)
if preds.ndim == 4 and preds.shape[1] == 16 and preds.shape[-1] != 16:
    preds = np.transpose(preds, (0, 2, 3, 1))
    print(f"preds shape (transposed to NHWT): {preds.shape}")

if preds.ndim != 4 or preds.shape[-1] != 16:
    raise ValueError(f"Expected preds shape like (N, H, W, 16), got {preds.shape}")

n, h, w, t_out = preds.shape
if t_out != 16:
    raise ValueError(f"Expected 16 forecast steps, got {t_out}")
if test_cpm25_hist.shape[0] != n or test_cpm25_hist.shape[2] != h or test_cpm25_hist.shape[3] != w:
    raise ValueError(
        'Spatial/sample mismatch between preds and test history: '
        f'preds={preds.shape}, test_hist={test_cpm25_hist.shape}'
    )

last_obs = test_cpm25_hist[:, -1, :, :]  # (N, H, W)

# 1) Horizon-1 consistency with latest observation
h1_pred = preds[..., 0]
rmse_h1_vs_last = float(np.sqrt(np.mean((h1_pred - last_obs) ** 2)))
mae_h1_vs_last = float(np.mean(np.abs(h1_pred - last_obs)))

# 2) Distance from persistence baseline over all forecast horizons
persistence = np.repeat(last_obs[..., None], preds.shape[-1], axis=-1)
rmse_all_vs_persistence = float(np.sqrt(np.mean((preds - persistence) ** 2)))
mae_all_vs_persistence = float(np.mean(np.abs(preds - persistence)))

# 3) Temporal smoothness inside forecast trajectory
step_delta = np.diff(preds, axis=-1)
mean_abs_step_change = float(np.mean(np.abs(step_delta)))

print('\nProxy scores (lower is generally better for RMSE/MAE):')
print(f"- RMSE(H+1 vs last observed):        {rmse_h1_vs_last:.4f}")
print(f"- MAE(H+1 vs last observed):         {mae_h1_vs_last:.4f}")
print(f"- RMSE(all horizons vs persistence): {rmse_all_vs_persistence:.4f}")
print(f"- MAE(all horizons vs persistence):  {mae_all_vs_persistence:.4f}")
print(f"- Mean abs step change (H to H+1):   {mean_abs_step_change:.4f}")

print('\nNote: Official competition score is computed on hidden future targets on Kaggle.')

In [ ]:
# ── Log proxy scores + prediction artifact to W&B ─────────────────────────────
if wandb_run is not None and wandb.run is not None:
    wandb.run.summary.update({
        'proxy/rmse_h1_vs_last_obs':        rmse_h1_vs_last,
        'proxy/mae_h1_vs_last_obs':         mae_h1_vs_last,
        'proxy/rmse_all_vs_persistence':    rmse_all_vs_persistence,
        'proxy/mae_all_vs_persistence':     mae_all_vs_persistence,
        'proxy/mean_abs_step_change':       mean_abs_step_change,
    })

    # Log preds.npy as a W&B artifact so it can be retrieved or compared later
    artifact = wandb.Artifact(
        name=f"preds-{wandb.run.id}",
        type="predictions",
        description="Test-set PM2.5 forecast (N, H, W, 16) in physical/normalized space",
        metadata={
            'preds_shape': list(preds.shape),
            'preds_path':  preds_path,
        },
    )
    artifact.add_file(preds_path)
    wandb.run.log_artifact(artifact)

    wandb.run.finish()
    print(f"Proxy scores and preds artifact logged to W&B run: {wandb_run.url}")
else:
    print("W&B run not active — skipping proxy score upload.")
